# ECG classification with Skip RNNs
This notebook implements window-level ECG classification using GRU and LSTM models with different state-update rules. It compares standard recurrent models with learned and random skipping, then extends the skip gate with heartbeat timing and shape features.

The input is the MLII lead from the MIT-BIH Arrhythmia Database. A positive 10-second window contains at least one annotated supraventricular ectopic, ventricular ectopic, or fusion beat. A negative label means none of those beats occurred; it does not necessarily indicate a normal rhythm.

## 1. Environment and data preparation
The setup cell resolves the project directory, installs missing Python packages, and imports the training and plotting modules. In Colab, PROJECT_FOLDER identifies the project directory relative to My Drive. Local execution uses the current working directory.

The 44 required recordings are checked against their published checksums. DOWNLOAD_MISSING controls whether missing or damaged recordings can be downloaded; otherwise, verification stops with an error. Signal preprocessing and feature extraction take place when the training datasets are constructed.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

PROJECT_FOLDER = 'ECG_Skip_Study'  # Relative to My Drive
DOWNLOAD_MISSING = False

try:
    from google.colab import drive
except ImportError:
    project = Path.cwd()  # Start local Jupyter in the folder containing this notebook.
else:
    drive.mount('/content/drive')
    project = Path('/content/drive/MyDrive') / PROJECT_FOLDER
scripts = project / 'Scripts'
if not (scripts / 'study.py').is_file():
    raise FileNotFoundError(f'Cannot find {scripts}. Check PROJECT_FOLDER and upload the whole submission.')
sys.path.insert(0, str(scripts.resolve()))
packages = {'torch':'torch>=2.1', 'numpy':'numpy', 'scipy':'scipy', 'sklearn':'scikit-learn', 'wfdb':'wfdb', 'pandas':'pandas', 'matplotlib':'matplotlib>=3.6'}
missing = [pkg for module, pkg in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)

import study
from figures import quality_and_cost, skip_strips
study.prepare_data(DOWNLOAD_MISSING)
print('Output folder:', study.RESULTS)

### Data and training protocol
All models use the same split: 17 training, 6 validation, and 21 test recordings. Lead MLII is selected by name because its channel position varies by recording. The signal is filtered causally, reduced to 180 Hz, and normalized using a running RMS.

The 10-second label intervals do not overlap. All models make their decision 0.5 seconds after each interval ends. ECG-only models observe 10.5 seconds of signal, so adjacent inputs overlap by 0.5 seconds. Physiology-guided models use 10 seconds of ECG plus heartbeat features available by the same decision time.

Each model uses a fixed configuration and seed 2. Recurrent models have one layer with 32 hidden units; learned skip gates use an update penalty of 1e-5. Training is capped at 60 epochs with early-stopping patience of 15 epochs. Skip models have five warm-up epochs before checkpoint selection. The checkpoint and classification threshold are selected on validation recordings and then held fixed for testing.

## 2. Standard, learned-skip, and random-skip models
This section trains six models sequentially: a standard, learned-skip, and random-skip variant for each of GRU and LSTM. The standard model updates its state at every sample. The Skip RNN gate learns when to update, while the random gate uses a fixed probability chosen to match the learned gate's average validation update rate. The resulting test update rates need not match exactly.

The table summarizes test accuracy, mean state updates, estimated computation per window, and the percentage of skipped samples. MFLOPs represent millions of floating-point operations, not measured runtime. The comparison follows the Skip RNN paper's classification setup on a different dataset rather than reproducing its original scores.

In [ ]:
study.run_section(study.PAPER)
display(study.results_table(study.PAPER).style.hide(axis='index').format(precision=2))

## 3. Physiology-guided skipping and additional controls
This section adds physiology-guided GRU and LSTM models. Their MLP skip gates combine the recurrent state with heartbeat timing and shape features and can refresh when a new beat is detected. Separate GRU and LSTM models use the same gate structure with the heartbeat inputs set to zero. These controls separate the use of a more flexible gate from the addition of explicit heartbeat information; the MLP controls updates rather than classifying the ECG directly.

A second random GRU targets the physiology-guided GRU's estimated validation computation, including heartbeat detection and feature extraction. A temporal convolutional network (TCN) provides a non-recurrent baseline with 9 levels and 16 channels per level. It has no recurrent update or skip percentage.

The extended table includes sensitivity, specificity, and within-record AUROC alongside accuracy and estimated computation. Within-record AUROC measures how well positive and negative windows are ranked within each recording, averaged over recordings containing both classes. The accompanying plot shows test accuracy and estimated computation for all models.

In [ ]:
study.run_section(study.EXTENSION)
display(study.results_table(study.PAPER + study.EXTENSION, extended=True).style.hide(axis='index').format(precision=3, na_rep='Not applicable'))
quality_and_cost()

## 4. State updates on an ECG segment
The plotting code selects a positive test window correctly classified by all six standard, learned-skip, and physiology-guided recurrent models. Each checkpoint is replayed to recover its actual update decisions on that window.

The GRU and LSTM figures use the same 10-second interval, with one row per model. Orange dots indicate updates and blue dots indicate skipped samples. The additional 0.5 seconds of ECG-only input is omitted from the display. This selected example illustrates update locations, not average performance. If no shared correctly classified positive window exists, no strip is drawn.

In [ ]:
skip_strips()

## 5. Training curves and update rates
This section plots the per-epoch histories of the standard, learned-skip, and physiology-guided GRU and LSTM models, along with the GRU and LSTM MLP-gate controls. The curves show the training objective, validation within-record AUROC, and validation update rate.

The training objective includes classification loss and, for learned skip models, the update penalty. The dotted AUROC line marks chance performance, and diamonds identify the validation-selected checkpoints used for testing. Each curve ends at the model's last recorded epoch.

Update rates are measured on validation windows after each training epoch. They are expressed as a percentage of each model's input length: 1,890 samples for ECG-only models and 1,800 for physiology-guided models. A value of 100% means that every sample updates the recurrent state.

In [ ]:
from importlib import reload
import figures

reload(figures).training_history()